In [ ]:
# 780k
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# =================================================================
# 1. CẤU HÌNH & TẢI DỮ LIỆU
# =================================================================
RAW_PATH = "../data/raw/"
SUBMISSION_PATH = "../submission/"
os.makedirs(SUBMISSION_PATH, exist_ok=True)

train_sales = pd.read_csv(f"{RAW_PATH}sales.csv", parse_dates=["Date"])
sample_sub = pd.read_csv(f"{RAW_PATH}sample_submission.csv", parse_dates=["Date"])

# Nền tảng 2021-2022 tinh khiết
train_sales = train_sales[train_sales['Date'].dt.year >= 2021].reset_index(drop=True)

# 🌟 TẠO MỤC TIÊU MỚI: TỶ LỆ GIÁ VỐN
train_sales['cogs_ratio'] = train_sales['COGS'] / train_sales['Revenue']

# =================================================================
# 2. FEATURE ENGINEERING (SÓNG FOURIER THUẦN TÚY)
# =================================================================
def create_pure_math_features(df):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['dayofyear'] = df['Date'].dt.dayofyear
    
    # Sóng Fourier (Linh hồn an toàn nhất)
    for k in range(1, 4):
        df[f'sin_y_{k}'] = np.sin(2 * np.pi * k * df['dayofyear'] / 365.25)
        df[f'cos_y_{k}'] = np.cos(2 * np.pi * k * df['dayofyear'] / 365.25)
        
    return df

train_df = create_pure_math_features(train_sales)
test_df = create_pure_math_features(sample_sub)

FEATURES = [col for col in train_df.columns if col not in ['Date', 'Revenue', 'COGS', 'cogs_ratio']]

# =================================================================
# 3. MÔ HÌNH HÓA (LÕI QUANTILE 0.5 - 350 CÂY)
# =================================================================
def get_quantile_predictions(target_col, is_ratio=False):
    y_raw = train_sales[target_col]
    
    et = ExtraTreesRegressor(n_estimators=350, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    rf = RandomForestRegressor(n_estimators=350, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    
    # Nếu là tỷ lệ (ratio) thì biên độ nhỏ, dùng learning rate nhỏ lại
    lr = 0.01 if is_ratio else 0.015
    hgb = HistGradientBoostingRegressor(loss='quantile', quantile=0.5, max_iter=400, learning_rate=lr, max_depth=7, random_state=42)
    
    print(f"🚀 Huấn luyện {target_col}...")
    et.fit(train_df[FEATURES], y_raw)
    rf.fit(train_df[FEATURES], y_raw)
    hgb.fit(train_df[FEATURES], y_raw)
    
    preds = np.median([
        et.predict(test_df[FEATURES]),
        rf.predict(test_df[FEATURES]),
        hgb.predict(test_df[FEATURES])
    ], axis=0)
    
    return preds

# Dự báo Doanh thu (Như cũ)
rev_raw = get_quantile_predictions("Revenue", is_ratio=False)
# Dự báo Tỷ lệ Giá vốn (Cách tiếp cận an toàn mới)
ratio_raw = get_quantile_predictions("cogs_ratio", is_ratio=True)

# =================================================================
# 4. CHỐT KẾT QUẢ - BẢO VỆ TỐI ĐA SỰ AN TOÀN
# =================================================================
submission = sample_sub.copy()

GROWTH_FACTOR = 1.38

# --- DOANH THU ---
submission['Revenue'] = rev_raw * GROWTH_FACTOR
submission['Revenue'] = np.clip(submission['Revenue'], a_min=0, a_max=11000000)

# --- GIÁ VỐN (COGS) TỰ NHIÊN ---
# Giới hạn tỷ lệ học được trong khoảng an toàn (ví dụ: từ 75% đến 98%) để tránh nhiễu
ratio_safe = np.clip(ratio_raw, a_min=0.75, a_max=0.98)

# Giá vốn tự động trượt theo quy mô doanh thu mới
submission['COGS'] = submission['Revenue'] * ratio_safe

submission = submission.round(2)
file_name = f"{SUBMISSION_PATH}submission.csv"
submission.to_csv(file_name, index=False)
print(f"\n✅ HOÀN TẤT LƯỢT 1/3! File tinh khiết nhất đã sẵn sàng tại: {file_name}")

🚀 Huấn luyện Revenue...
🚀 Huấn luyện cogs_ratio...

✅ HOÀN TẤT LƯỢT 1/3! File tinh khiết nhất đã sẵn sàng tại: ../submission/submission.csv


C:\Users\Admin\AppData\Local\Temp\ipykernel_15204\429523766.py:93: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  submission = submission.round(2)


In [ ]:
# 777k
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# =================================================================
# 1. CẤU HÌNH & TẢI DỮ LIỆU (BẢO TOÀN NỀN TẢNG TINH KHIẾT)
# =================================================================
RAW_PATH = "../data/raw/"
SUBMISSION_PATH = "../submission/"
os.makedirs(SUBMISSION_PATH, exist_ok=True)

train_sales = pd.read_csv(f"{RAW_PATH}sales.csv", parse_dates=["Date"])
sample_sub = pd.read_csv(f"{RAW_PATH}sample_submission.csv", parse_dates=["Date"])

train_sales = train_sales[train_sales['Date'].dt.year >= 2021].reset_index(drop=True)

# 🌟 Vẫn là mục tiêu vàng: TỶ LỆ GIÁ VỐN
train_sales['cogs_ratio'] = train_sales['COGS'] / train_sales['Revenue']

# =================================================================
# 2. FEATURE ENGINEERING (GIỮ CHẶT SÓNG FOURIER, KHÔNG THÊM RÁC)
# =================================================================
def create_pure_math_features(df):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['dayofyear'] = df['Date'].dt.dayofyear
    
    for k in range(1, 4):
        df[f'sin_y_{k}'] = np.sin(2 * np.pi * k * df['dayofyear'] / 365.25)
        df[f'cos_y_{k}'] = np.cos(2 * np.pi * k * df['dayofyear'] / 365.25)
        
    return df

train_df = create_pure_math_features(train_sales)
test_df = create_pure_math_features(sample_sub)

FEATURES = [col for col in train_df.columns if col not in ['Date', 'Revenue', 'COGS', 'cogs_ratio']]

# =================================================================
# 3. MÔ HÌNH HÓA (VI CHỈNH NHẸ NHÀNG ĐỂ BẮT ĐÁY/ĐỈNH TỐT HƠN)
# =================================================================
def get_quantile_predictions(target_col, is_ratio=False):
    y_raw = train_sales[target_col]
    
    # 🌟 Nâng lên 400 cây để tăng độ phân giải
    et = ExtraTreesRegressor(n_estimators=400, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    rf = RandomForestRegressor(n_estimators=400, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    
    # Giảm learning rate nhẹ để tránh bị trượt đỉnh
    lr = 0.01 if is_ratio else 0.012
    hgb = HistGradientBoostingRegressor(loss='quantile', quantile=0.5, max_iter=450, learning_rate=lr, max_depth=7, random_state=42)
    
    print(f"🚀 Huấn luyện độ phân giải cao cho {target_col}...")
    et.fit(train_df[FEATURES], y_raw)
    rf.fit(train_df[FEATURES], y_raw)
    hgb.fit(train_df[FEATURES], y_raw)
    
    preds = np.median([
        et.predict(test_df[FEATURES]),
        rf.predict(test_df[FEATURES]),
        hgb.predict(test_df[FEATURES])
    ], axis=0)
    
    return preds

# Dự báo độc lập
rev_raw = get_quantile_predictions("Revenue", is_ratio=False)
ratio_raw = get_quantile_predictions("cogs_ratio", is_ratio=True)

# =================================================================
# 4. CHỐT KẾT QUẢ - BẢO VỆ TỐI ĐA THÀNH QUẢ 780K
# =================================================================
submission = sample_sub.copy()

GROWTH_FACTOR = 1.38

# --- DOANH THU ---
submission['Revenue'] = rev_raw * GROWTH_FACTOR
submission['Revenue'] = np.clip(submission['Revenue'], a_min=0, a_max=11000000)

# --- GIÁ VỐN (COGS) TỰ NHIÊN ---
# 🌟 CÚ ĐÁNH QUYẾT ĐỊNH: Nới biên độ từ 0.98 lên 1.00 (Chấp nhận ngày hòa vốn)
ratio_safe = np.clip(ratio_raw, a_min=0.70, a_max=1.00)

# Giá vốn bám sát doanh thu
submission['COGS'] = submission['Revenue'] * ratio_safe

submission = submission.round(2)
file_name = f"{SUBMISSION_PATH}submission.csv"
submission.to_csv(file_name, index=False)
print(f"\n✅ HOÀN TẤT LƯỢT 2/3! Bản tinh chỉnh đã sẵn sàng tại: {file_name}")

🚀 Huấn luyện độ phân giải cao cho Revenue...
🚀 Huấn luyện độ phân giải cao cho cogs_ratio...

✅ HOÀN TẤT LƯỢT 2/3! Bản tinh chỉnh đã sẵn sàng tại: ../submission/submission.csv


C:\Users\Admin\AppData\Local\Temp\ipykernel_15204\968958229.py:91: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  submission = submission.round(2)


In [ ]:
# 777k
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor

# =================================================================
# 1. CẤU HÌNH & TẢI DỮ LIỆU
# =================================================================
RAW_PATH = "../data/raw/"
SUBMISSION_PATH = "../submission/"
os.makedirs(SUBMISSION_PATH, exist_ok=True)

train_sales = pd.read_csv(f"{RAW_PATH}sales.csv", parse_dates=["Date"])
sample_sub = pd.read_csv(f"{RAW_PATH}sample_submission.csv", parse_dates=["Date"])

train_sales = train_sales[train_sales['Date'].dt.year >= 2021].reset_index(drop=True)

# MỤC TIÊU VÀNG: TỶ LỆ GIÁ VỐN
train_sales['cogs_ratio'] = train_sales['COGS'] / train_sales['Revenue']

# =================================================================
# 2. FEATURE ENGINEERING (SÓNG FOURIER NGUYÊN BẢN)
# =================================================================
def create_pure_math_features(df):
    df = df.copy()
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['dayofyear'] = df['Date'].dt.dayofyear
    
    for k in range(1, 4):
        df[f'sin_y_{k}'] = np.sin(2 * np.pi * k * df['dayofyear'] / 365.25)
        df[f'cos_y_{k}'] = np.cos(2 * np.pi * k * df['dayofyear'] / 365.25)
        
    return df

train_df = create_pure_math_features(train_sales)
test_df = create_pure_math_features(sample_sub)

FEATURES = [col for col in train_df.columns if col not in ['Date', 'Revenue', 'COGS', 'cogs_ratio']]

# =================================================================
# 3. MÔ HÌNH HÓA (ĐẨY ĐỘ PHÂN GIẢI LÊN MAX: 500 CÂY)
# =================================================================
def get_quantile_predictions(target_col, is_ratio=False):
    y_raw = train_sales[target_col]
    
    # Kích hoạt 500 cây để vắt kiệt quy luật cuối cùng
    et = ExtraTreesRegressor(n_estimators=500, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    rf = RandomForestRegressor(n_estimators=500, criterion='absolute_error', max_depth=9, random_state=42, n_jobs=-1)
    
    lr = 0.01 if is_ratio else 0.012
    hgb = HistGradientBoostingRegressor(loss='quantile', quantile=0.5, max_iter=500, learning_rate=lr, max_depth=7, random_state=42)
    
    print(f"🚀 Huấn luyện Lõi 500 Cây cho {target_col}...")
    et.fit(train_df[FEATURES], y_raw)
    rf.fit(train_df[FEATURES], y_raw)
    hgb.fit(train_df[FEATURES], y_raw)
    
    preds = np.median([
        et.predict(test_df[FEATURES]),
        rf.predict(test_df[FEATURES]),
        hgb.predict(test_df[FEATURES])
    ], axis=0)
    
    return preds

rev_raw = get_quantile_predictions("Revenue", is_ratio=False)
ratio_raw = get_quantile_predictions("cogs_ratio", is_ratio=True)

# =================================================================
# 4. CHỐT KẾT QUẢ - BẢN NỘP CUỐI CÙNG TRONG NGÀY
# =================================================================
submission = sample_sub.copy()

GROWTH_FACTOR = 1.38

# --- DOANH THU (BẤT DI BẤT DỊCH) ---
submission['Revenue'] = rev_raw * GROWTH_FACTOR
submission['Revenue'] = np.clip(submission['Revenue'], a_min=0, a_max=11000000)

# --- GIÁ VỐN (MỞ KHÓA TRỢ GIÁ KHUYẾN MÃI) ---
# Cho phép AI dự báo lỗ tới 5% (1.05) hoặc biên lợi nhuận dày 35% (0.65)
ratio_safe = np.clip(ratio_raw, a_min=0.65, a_max=1.05)
submission['COGS'] = submission['Revenue'] * ratio_safe

submission = submission.round(2)
file_name = f"{SUBMISSION_PATH}submission.csv"
submission.to_csv(file_name, index=False)
print(f"\n✅ HOÀN TẤT! Phát đạn cuối cùng đã lên nòng tại: {file_name}")

🚀 Huấn luyện Lõi 500 Cây cho Revenue...
🚀 Huấn luyện Lõi 500 Cây cho cogs_ratio...

✅ HOÀN TẤT! Phát đạn cuối cùng đã lên nòng tại: ../submission/submission.csv


C:\Users\Admin\AppData\Local\Temp\ipykernel_15204\656627018.py:87: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  submission = submission.round(2)
